In [12]:
import pandas as pd
import numpy as np
import polars as pl

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("polars:", pl.__version__)

pandas: 3.0.3
numpy: 2.4.6
polars: 1.42.0


In [2]:
related_files = {
    'bureau': '../data/raw/bureau.csv',
    'bureau_balance': '../data/raw/bureau_balance.csv',
    'previous_application': '../data/raw/previous_application.csv',
    'POS_CASH_balance': '../data/raw/POS_CASH_balance.csv',
    'credit_card_balance': '../data/raw/credit_card_balance.csv',
    'installments_payments': '../data/raw/installments_payments.csv',
}

# Fast reconnaissance pass: schema and row count for all 6 tables via polars,
# without committing to loading any of them into pandas yet.
for name, path in related_files.items():
    schema = pl.scan_csv(path).collect_schema()
    n_rows = pl.scan_csv(path).select(pl.len()).collect().item()
    print(f'=== {name} ===')
    print('Rows:', n_rows, '| Columns:', len(schema))
    print('Column names:', list(schema.names()))
    print()

=== bureau ===
Rows: 1716428 | Columns: 17
Column names: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

=== bureau_balance ===
Rows: 27299925 | Columns: 3
Column names: ['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

=== previous_application ===
Rows: 1670214 | Columns: 37
Column names: ['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_R

In [3]:
# Only bureau.csv gets fully loaded into pandas at this stage,
# consistent with the sliced pipeline strategy (bureau-first).
bureau = pd.read_csv('../data/raw/bureau.csv')
print(bureau.shape)

(1716428, 17)


In [4]:
# Check dtypes to confirm pandas interpreted each column correctly
# before deciding aggregation functions (numeric vs categorical).
bureau.dtypes

SK_ID_CURR                  int64
SK_ID_BUREAU                int64
CREDIT_ACTIVE                 str
CREDIT_CURRENCY               str
DAYS_CREDIT                 int64
CREDIT_DAY_OVERDUE          int64
DAYS_CREDIT_ENDDATE       float64
DAYS_ENDDATE_FACT         float64
AMT_CREDIT_MAX_OVERDUE    float64
CNT_CREDIT_PROLONG          int64
AMT_CREDIT_SUM            float64
AMT_CREDIT_SUM_DEBT       float64
AMT_CREDIT_SUM_LIMIT      float64
AMT_CREDIT_SUM_OVERDUE    float64
CREDIT_TYPE                   str
DAYS_CREDIT_UPDATE          int64
AMT_ANNUITY               float64
dtype: object

In [5]:
# Load the full data dictionary again (this notebook is separate from 01_eda.ipynb)
# and filter to bureau.csv specifically.
desc = pd.read_csv('../data/raw/HomeCredit_columns_description.csv', encoding='latin-1')
bureau_desc = desc[desc['Table'] == 'bureau.csv'][['Row', 'Description']]
print(bureau_desc.to_string(index=False))

                   Row                                                                                                         Description
            SK_ID_CURR ID of loan in our sample - one loan in our sample can have 0,1,2 or more related previous credits in credit bureau 
          SK_BUREAU_ID           Recoded ID of previous Credit Bureau credit related to our loan (unique coding for each loan application)
         CREDIT_ACTIVE                                                                   Status of the Credit Bureau (CB) reported credits
       CREDIT_CURRENCY                                                                        Recoded currency of the Credit Bureau credit
           DAYS_CREDIT                                  How many days before current application did client apply for Credit Bureau credit
    CREDIT_DAY_OVERDUE                      Number of days past due on CB credit at the time of application for related loan in our sample
   DAYS_CREDIT_ENDDATE     

In [7]:
print("bureau, linhas por cliente (média):", bureau.groupby('SK_ID_CURR').size().mean().round(2))

bureau, linhas por cliente (média): 5.61


In [8]:
bureau['CREDIT_ACTIVE'].unique()

<StringArray>
['Closed', 'Active', 'Sold', 'Bad debt']
Length: 4, dtype: str

In [9]:
# CREDIT_CURRENCY: check if it's near-constant (low discriminative power expected)
print(bureau['CREDIT_CURRENCY'].value_counts())
print(bureau['CREDIT_CURRENCY'].value_counts(normalize=True).round(4))

CREDIT_CURRENCY
currency 1    1715020
currency 2       1224
currency 3        174
currency 4         10
Name: count, dtype: int64
CREDIT_CURRENCY
currency 1    0.9992
currency 2    0.0007
currency 3    0.0001
currency 4    0.0000
Name: proportion, dtype: float64


In [10]:
# CREDIT_TYPE: preview of categories before deciding nunique + most frequent category count
print(bureau['CREDIT_TYPE'].value_counts())
print()
print("Número de categorias únicas:", bureau['CREDIT_TYPE'].nunique())

CREDIT_TYPE
Consumer credit                                 1251615
Credit card                                      402195
Car loan                                          27690
Mortgage                                          18391
Microloan                                         12413
Loan for business development                      1975
Another type of loan                               1017
Unknown type of loan                                555
Loan for working capital replenishment              469
Cash loan (non-earmarked)                            56
Real estate loan                                     27
Loan for the purchase of equipment                   19
Loan for purchase of shares (margin lending)          4
Mobile operator loan                                  1
Interbank credit                                      1
Name: count, dtype: int64

Número de categorias únicas: 15


## Bureau.csv aggregation to SK_ID_CURR level

Systematic, column-by-column review of all 17 columns in bureau.csv, deciding an aggregation
treatment for each one instead of an intuitive partial selection.

**Aggregation criteria applied:**
- `sum`: used when the total aggregated exposure matters (for example, total credit amount
  across all external credits)
- `mean`: used to capture a typical per-credit profile, normalized by the number of credits
- `max`: used to preserve a worst-case signal that a mean would dilute (for example, the
  single worst overdue event, even if only one out of several credits had a problem)
- `count` / `nunique`: used for occurrence counts or category diversity

**Column-level decisions:**
- `CREDIT_CURRENCY` excluded from aggregation. Confirmed via value_counts that 99.92% of
  records are concentrated in a single category, leaving no practical discriminative power.
- `CREDIT_ACTIVE` (4 categories: Active, Closed, Sold, Bad debt) aggregated as a count per
  category via crosstab, since the low cardinality makes this equivalent to one-hot encoding
  without needing WoE at this stage.
- `CREDIT_TYPE` (15 categories, heavily concentrated in Consumer credit and Credit card)
  aggregated two ways: `nunique` (credit type diversification) and the count of each client's
  most frequent category (credit type concentration). These are complementary signals, not
  redundant.
- Note: the official Kaggle data dictionary names the bureau ID column `SK_BUREAU_ID`, but
  the actual CSV column is `SK_ID_BUREAU`. This is a documentation typo, not a data issue,
  confirmed by cross-checking `bureau.dtypes` against the dictionary.

**Deferred to later stages:**
- WoE encoding for `CREDIT_TYPE` and `CREDIT_ACTIVE` is not applied here, since WoE requires
  the TARGET variable, which only exists in application_train. WoE calculation happens after
  merging bureau_agg with application_train, and only for the logistic regression scorecard,
  not for the XGBoost model.

**Final scheme:**

| Column | Treatment |
|---|---|
| SK_ID_BUREAU | count |
| CREDIT_ACTIVE | count per category (Active, Closed, Sold, Bad debt) |
| CREDIT_CURRENCY | excluded (99.92% concentrated in a single category) |
| DAYS_CREDIT | mean, min |
| CREDIT_DAY_OVERDUE | max, mean |
| DAYS_CREDIT_ENDDATE | mean |
| DAYS_ENDDATE_FACT | mean |
| AMT_CREDIT_MAX_OVERDUE | max, mean |
| CNT_CREDIT_PROLONG | sum |
| AMT_CREDIT_SUM | sum, mean |
| AMT_CREDIT_SUM_DEBT | sum, mean |
| AMT_CREDIT_SUM_LIMIT | sum, mean |
| AMT_CREDIT_SUM_OVERDUE | sum, max |
| CREDIT_TYPE | nunique, count of most frequent category |
| DAYS_CREDIT_UPDATE | mean, max |
| AMT_ANNUITY | sum, mean |

In [11]:
# Column-by-column, systematic aggregation of bureau.csv to SK_ID_CURR level.
# CREDIT_CURRENCY excluded: 99.92% concentrated in a single category, no discriminative power.
# Note: the official Kaggle dictionary names this table's ID column "SK_BUREAU_ID",
# but the actual CSV uses "SK_ID_BUREAU". Documentation typo, not a data issue.

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
    BUREAU_DAYS_CREDIT_MIN=('DAYS_CREDIT', 'min'),
    BUREAU_CREDIT_DAY_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_CREDIT_DAY_OVERDUE_MEAN=('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_DAYS_CREDIT_ENDDATE_MEAN=('DAYS_CREDIT_ENDDATE', 'mean'),
    BUREAU_DAYS_ENDDATE_FACT_MEAN=('DAYS_ENDDATE_FACT', 'mean'),
    BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN=('AMT_CREDIT_MAX_OVERDUE', 'mean'),
    BUREAU_CNT_CREDIT_PROLONG_SUM=('CNT_CREDIT_PROLONG', 'sum'),
    BUREAU_AMT_CREDIT_SUM_SUM=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_CREDIT_SUM_MEAN=('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AMT_CREDIT_SUM_DEBT_SUM=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_AMT_CREDIT_SUM_DEBT_MEAN=('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_AMT_CREDIT_SUM_LIMIT_SUM=('AMT_CREDIT_SUM_LIMIT', 'sum'),
    BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN=('AMT_CREDIT_SUM_LIMIT', 'mean'),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX=('AMT_CREDIT_SUM_OVERDUE', 'max'),
    BUREAU_CREDIT_TYPE_NUNIQUE=('CREDIT_TYPE', 'nunique'),
    BUREAU_DAYS_CREDIT_UPDATE_MEAN=('DAYS_CREDIT_UPDATE', 'mean'),
    BUREAU_DAYS_CREDIT_UPDATE_MAX=('DAYS_CREDIT_UPDATE', 'max'),
    BUREAU_AMT_ANNUITY_SUM=('AMT_ANNUITY', 'sum'),
    BUREAU_AMT_ANNUITY_MEAN=('AMT_ANNUITY', 'mean'),
).reset_index()

# CREDIT_ACTIVE: count per category (Active, Closed, Sold, Bad debt).
credit_active_counts = pd.crosstab(bureau['SK_ID_CURR'], bureau['CREDIT_ACTIVE'])
credit_active_counts.columns = [f'BUREAU_CREDIT_ACTIVE_{c.upper().replace(" ", "_")}_COUNT' for c in credit_active_counts.columns]
credit_active_counts = credit_active_counts.reset_index()

# CREDIT_TYPE: count of each client's most frequent credit type (concentration signal).
credit_type_mode_count = bureau.groupby('SK_ID_CURR')['CREDIT_TYPE'].agg(
    lambda x: x.value_counts().iloc[0]
).reset_index(name='BUREAU_CREDIT_TYPE_MODE_COUNT')

bureau_agg = bureau_agg.merge(credit_active_counts, on='SK_ID_CURR', how='left')
bureau_agg = bureau_agg.merge(credit_type_mode_count, on='SK_ID_CURR', how='left')

print(bureau_agg.shape)
bureau_agg.head()

(305811, 29)


,SK_ID_CURR,BUREAU_CREDIT_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_ENDDATE_FACT_MEAN,BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX,BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN,...,BUREAU_CREDIT_TYPE_NUNIQUE,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT,BUREAU_CREDIT_ACTIVE_CLOSED_COUNT,BUREAU_CREDIT_ACTIVE_SOLD_COUNT,BUREAU_CREDIT_TYPE_MODE_COUNT
0,100001,7,-735.000000,-1572,0,0.0,82.428571,-825.500000,NaN,NaN,...,1,-93.142857,-6,24817.5,3545.357143,3,0,4,0,7
1,100002,8,-874.000000,-1437,0,0.0,-349.000000,-697.500000,5043.645,1681.029,...,2,-499.875000,-7,0.0,0.000000,2,0,6,0,4
2,100003,4,-1400.750000,-2586,0,0.0,-544.500000,-1097.333333,0.000,0.000,...,2,-816.000000,-43,0.0,NaN,1,0,3,0,2
3,100004,2,-867.000000,-1326,0,0.0,-488.500000,-532.500000,0.000,0.000,...,1,-532.000000,-382,0.0,NaN,0,0,2,0,2
4,100005,3,-190.666667,-373,0,0.0,439.333333,-123.000000,0.000,0.000,...,2,-54.333333,-11,4261.5,1420.500000,2,0,1,0,2


In [15]:
# Merge application_train with bureau_agg, save as intermediate artifact
# for the modeling notebook to consume without re-running aggregation.

application_train = pd.read_csv('../data/raw/application_train.csv')
print("application_train shape:", application_train.shape)

application_train shape: (307511, 122)


In [16]:
# Replicate DAYS_EMPLOYED sentinel treatment and readability columns from 01_eda.ipynb.
# These transformations were done in-memory in the EDA notebook and are not persisted
# in application_train.csv, so they must be reapplied here before merging with bureau_agg.

# DAYS_EMPLOYED sentinel (365243): flag first, then convert to NaN.
application_train['DAYS_EMPLOYED_ANOM'] = (application_train['DAYS_EMPLOYED'] == 365243).astype('int8')
application_train['DAYS_EMPLOYED'] = application_train['DAYS_EMPLOYED'].replace(365243, np.nan)

# Readability columns in years, original DAYS_* columns preserved.
application_train['AGE_YEARS'] = (-application_train['DAYS_BIRTH'] / 365).round(1)
application_train['YEARS_EMPLOYED'] = (-application_train['DAYS_EMPLOYED'] / 365).round(1)
application_train['YEARS_REGISTRATION'] = (-application_train['DAYS_REGISTRATION'] / 365).round(1)
application_train['YEARS_ID_PUBLISH'] = (-application_train['DAYS_ID_PUBLISH'] / 365).round(1)
application_train['YEARS_LAST_PHONE_CHANGE'] = (-application_train['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)

# Defragment after multiple individual column insertions.
application_train = application_train.copy()

print(application_train.shape)
print(application_train['DAYS_EMPLOYED_ANOM'].value_counts())

C:\Users\vitor\AppData\Local\Temp\ipykernel_24040\2116203301.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['DAYS_EMPLOYED_ANOM'] = (application_train['DAYS_EMPLOYED'] == 365243).astype('int8')
C:\Users\vitor\AppData\Local\Temp\ipykernel_24040\2116203301.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['AGE_YEARS'] = (-application_train['DAYS_BIRTH'] / 365).round(1)
C:\Users\vitor\AppData\Local\Temp\ipykernel_24040\2116203301.py:11: PerformanceWarning: DataFrame is highly fragmented. 

(307511, 128)
DAYS_EMPLOYED_ANOM
0    252137
1     55374
Name: count, dtype: int64


In [17]:
# Merging
train_bureau = application_train.merge(bureau_agg, on='SK_ID_CURR', how='left')

print("train_bureau shape:", train_bureau.shape)
train_bureau.to_csv('../data/processed/train_bureau.csv', index=False)

train_bureau shape: (307511, 156)


In [18]:
# Confirm the expected NaN pattern for clients without bureau history.
bureau_cols = [c for c in train_bureau.columns if c.startswith('BUREAU_')]
no_bureau_history = train_bureau[bureau_cols].isnull().all(axis=1).sum()

print("Clientes sem nenhum histórico de bureau (todas as colunas BUREAU_* nulas):", no_bureau_history)
print("Esperado:", 307511 - 305811)

Clientes sem nenhum histórico de bureau (todas as colunas BUREAU_* nulas): 44020
Esperado: 1700


In [19]:
# Isolate merge non-matches specifically, using a column that can only be
# NaN due to the left join not finding a match (count never returns NaN
# for a client who does have bureau history).
no_match = train_bureau['BUREAU_CREDIT_COUNT'].isnull().sum()
print("Clientes sem match no merge (BUREAU_CREDIT_COUNT nulo):", no_match)
print("Esperado:", 307511 - 305811)

Clientes sem match no merge (BUREAU_CREDIT_COUNT nulo): 44020
Esperado: 1700


In [20]:
# Test the hypothesis: does bureau.csv cover clients beyond application_train
# (i.e., does it include application_test clients too)?

bureau_client_ids = set(bureau['SK_ID_CURR'].unique())
train_client_ids = set(application_train['SK_ID_CURR'].unique())

only_in_bureau_not_in_train = bureau_client_ids - train_client_ids
train_not_in_bureau = train_client_ids - bureau_client_ids

print("IDs em bureau que NÃO estão em application_train:", len(only_in_bureau_not_in_train))
print("IDs em application_train que NÃO estão em bureau (sem histórico):", len(train_not_in_bureau))

IDs em bureau que NÃO estão em application_train: 42320
IDs em application_train que NÃO estão em bureau (sem histórico): 44020
